# Part 2: same seed, same beginning
**Text watermarking with a real LLM** · Mayank Raj and Shreya Agrahari

In Part 1, we gave a program four words to choose from. This time, a small language model will choose the words.

We'll generate twice, compare the results, then add a preference and repeat. At the end, paste text into a box and check for our watermark.

Run one cell at a time with **Shift + Enter**. The setup cells are provided. Our first experiment starts at step 3.

## 1. The values you can change
`STARTING_WORDS` is the beginning of our story. It has five words.

`SEED` controls the random choices made during sampling. We'll reset it before each run.

`NEW_TOKENS` limits the length of the continuation. A token can be a word, part of a word, or punctuation. Leave this at 160 for the first run.

In [ ]:
SEED = 42
STARTING_WORDS = "The old lighthouse suddenly went"
NEW_TOKENS = 160

## 2. Load the tools and model
Run the installer first. If it asks you to restart, restart the notebook session before continuing, then run from the top. The installer will reuse the installed versions after the restart.

We use CPU execution. On Colab, Linux and Windows, the installer selects the official PyTorch CPU wheel. On macOS, it uses the standard wheel and our model still runs on CPU. The text lab does not need torchvision, torchaudio or librosa; the installer removes these optional packages if present so they cannot break model imports.

The first model load needs internet and downloads about 0.7 GB of weights. No paid API key or Hugging Face token is needed for this public model. You can ignore Colab's missing `HF_TOKEN` warning.

In [ ]:
#@title Install the tools (run once; restart if requested)
import platform, subprocess, sys
from importlib import metadata
from packaging.specifiers import SpecifierSet

TORCH_VERSION = "2.6.0" if platform.system() == "Darwin" else "2.6.0+cpu"
PINS = {
    "torch": TORCH_VERSION,
    "transformers": "4.51.3",
    "tokenizers": "0.21.4",
    "huggingface-hub": "0.36.2",
    "ipywidgets": "8.1.7",
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

if globals().get("_WORKSHOP_RESTART_REQUIRED", False):
    print("Restart the session, then run from the top. Do not continue yet.")
else:
    optional = [name for name in ("torchvision", "torchaudio", "librosa")
                if installed_version(name) is not None]
    safetensors_version = installed_version("safetensors")
    safetensors_ok = (safetensors_version is not None and
                     safetensors_version in SpecifierSet(">=0.5,<1"))
    changes_needed = (bool(optional) or not safetensors_ok or
                      any(installed_version(name) != version
                          for name, version in PINS.items()))
    if changes_needed:
        # Keep the restart guard set even if an installation fails partway.
        _WORKSHOP_RESTART_REQUIRED = True
        if optional:
            pip("uninstall", "-y", *optional)
        if installed_version("torch") != TORCH_VERSION:
            torch_args = ["install", "--no-cache-dir", "torch==" + TORCH_VERSION]
            if platform.system() != "Darwin":
                torch_args += ["--index-url", "https://download.pytorch.org/whl/cpu"]
            pip(*torch_args)
        pip("install", "--no-cache-dir",
            *[name + "==" + version for name, version in PINS.items()
              if name != "torch"], "safetensors>=0.5,<1")
        print("Installed. Restart the session, then run from the top.")
    else:
        _WORKSHOP_RESTART_REQUIRED = False
        print("Dependencies ready. Continue to the next cell.")

In [ ]:
#@title Check the setup and enable repeatable CPU execution
if globals().get("_WORKSHOP_RESTART_REQUIRED", False):
    raise RuntimeError("Restart the session, then run from the top before continuing.")

import importlib.util, platform
for package in ("torchvision", "torchaudio", "librosa"):
    if importlib.util.find_spec(package) is not None:
        raise RuntimeError("Run the installer and restart the session first: " + package)

import os, math, torch, transformers
expected_torch = "2.6.0" if platform.system() == "Darwin" else "2.6.0+cpu"
if torch.__version__ != expected_torch or transformers.__version__ != "4.51.3":
    raise RuntimeError("Run the installer, restart the session, then run from the top.")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, set_seed
from transformers import WatermarkingConfig, WatermarkLogitsProcessor
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)
print("PASS: text model and watermark imports work.")
print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)

In [ ]:
#@title Load the fixed model version (run once)
MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, revision=REVISION, torch_dtype=torch.float32,
    attn_implementation="eager", use_safetensors=True).cpu().eval()
print("Model ready")

## 3. Give the model our beginning
The **tokenizer** turns text into token numbers the model understands. We keep the length of the beginning so we can separate it from the new text later.

If you change `STARTING_WORDS` in step 1, rerun this cell before generating again.

In [ ]:
inputs = tokenizer(STARTING_WORDS, return_tensors="pt")
beginning_length = inputs.input_ids.shape[1]
print("Beginning:", STARTING_WORDS)
print("Tokens in the beginning:", beginning_length)

Here are the generation settings. A **dictionary** stores named settings inside `{}`. Later, `**settings` passes all of them to the model.

`do_sample=True` means the model samples a token from its predicted probabilities. This is the choice the seed controls. Keep these settings fixed while comparing the two runs.

In [ ]:
settings = {
    "max_new_tokens": NEW_TOKENS,
    "min_new_tokens": min(120, NEW_TOKENS),
    "do_sample": True, "temperature": 0.8, "top_k": 0,
    "pad_token_id": tokenizer.eos_token_id,
}

## 4. Generate once
The model predicts from the preceding tokens. The sampler then chooses the next token.

`set_seed(SEED)` resets the random choices. `model.generate()` produces the continuation. `tokenizer.decode()` turns its token numbers back into text.

The `with` line tells PyTorch we're generating, not training. The indented line belongs inside it. The slice `[beginning_length:]` skips the starting words when we decode.

In [ ]:
set_seed(SEED)
with torch.inference_mode():
    first = model.generate(**inputs, **settings)
first_text = tokenizer.decode(first[0][beginning_length:], skip_special_tokens=True)
print("RUN 1")
print(STARTING_WORDS + first_text)

## 5. Run the same thing again
Same model. Same beginning. Same settings. Reset the same seed too.

Predict the result before running this cell.

In [ ]:
set_seed(SEED)
with torch.inference_mode():
    second = model.generate(**inputs, **settings)
second_text = tokenizer.decode(second[0][beginning_length:], skip_special_tokens=True)
print("RUN 2")
print(STARTING_WORDS + second_text)

Compare the token sequences, not just how the paragraphs look. `torch.equal()` checks whether the token numbers are identical.

In [ ]:
print("Exactly the same tokens?", torch.equal(first, second))
print("Exactly the same text?", first_text == second_text)

Both should be **True** in this controlled notebook.

**Try it:** change `SEED` to `99`, rerun that variable cell, then rerun steps 4 and 5. Those two runs should also match each other. A different seed can produce a different continuation.

Next, try five different starting words. Rerun steps 1, 3, 4 and 5. Each next-token prediction uses the text that came before it.

This shows repeatability with the same setup. It does not promise identical results across different machines, model versions or generation settings.

## 6. Add a watermark preference
In Part 1, we favoured two fixed words. Here, the watermark rule chooses preferred tokens using a **watermark key** and the previous token. The preferred set changes as the text grows.

The key has a different job from `SEED`: the key defines the preference; the seed repeats the sampling choices. `bias` controls how strongly we favour the preferred tokens.

We keep the same model. This changes token selection during generation.

In [ ]:
WATERMARK_KEY = 15485863
watermark = WatermarkingConfig(
    hashing_key=WATERMARK_KEY, bias=2.0,
    greenlist_ratio=0.25,
    seeding_scheme="lefthash", context_width=1,
)

This is the same generation call with one new option: `watermarking_config=watermark`.

In [ ]:
set_seed(SEED)
with torch.inference_mode():
    marked_first = model.generate(**inputs, **settings, watermarking_config=watermark)
marked_text = tokenizer.decode(marked_first[0][beginning_length:], skip_special_tokens=True)
print("WATERMARKED RUN 1")
print(STARTING_WORDS + marked_text)

Repeat the watermarked run too. Changing the preference does not stop us from making the run repeatable.

In [ ]:
set_seed(SEED)
with torch.inference_mode():
    marked_second = model.generate(**inputs, **settings, watermarking_config=watermark)
marked_again = tokenizer.decode(marked_second[0][beginning_length:], skip_special_tokens=True)
print("WATERMARKED RUN 2")
print(STARTING_WORDS + marked_again)

In [ ]:
print("Watermarked runs match?", torch.equal(marked_first, marked_second))
print("Normal and marked text match?", first_text == marked_text)

For the starting example, expect **True**, then **False**. The marked runs repeat each other, but the preference changes the continuation compared with the normal run.

Read the two versions. Did the stronger preference also affect how natural the text sounds? This small model can produce awkward writing.

## 7. Check for our watermark
The detector uses the same tokenizer and watermark rule. It reconstructs which tokens were preferred and checks how often they appeared. It does not need the original sampling seed or a saved copy of the generated paragraph.

The next cell is provided detector code. **Run it; you don't need to retype it.** It ignores repeated token pairs and returns no score when there are too few different pairs. The score threshold is a classroom example, not a calibrated authorship test.

In [ ]:
#@title Provided detector: run this cell
def check_watermark(text, config=watermark):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) > 2048:
        return "Please use a shorter passage (up to 2,048 tokens)."
    pairs = set(zip(ids, ids[1:]))
    if len(pairs) < 40:
        return "Too little varied text. Try a longer passage."
    preference = WatermarkLogitsProcessor(
        model.config.vocab_size, "cpu", bias=1.0,
        greenlist_ratio=config.greenlist_ratio, hashing_key=config.hashing_key,
        seeding_scheme="lefthash", context_width=1)
    hits = 0
    for previous, current in pairs:
        scores = preference(torch.tensor([[previous]]),
                            torch.zeros(1, model.config.vocab_size))
        hits += int(scores[0, current] > 0)
    expected = len(pairs) * config.greenlist_ratio
    spread = math.sqrt(expected * (1 - config.greenlist_ratio))
    score = (hits - expected) / spread
    verdict = "Watermark detected" if score > 3 else "Watermark not detected"
    return f"{verdict} | preferred pairs: {hits}/{len(pairs)} | score: {score:.2f}"

First, check the two continuations we just generated. We leave the five starting words out of the detector input.

In [ ]:
print("Normal:", check_watermark(first_text))
print("Marked:", check_watermark(marked_text))

## 8. Paste some text
Run this provided interface cell. Paste a continuation into the box, then press **Check text**.

Try the marked passage, the normal passage, then an edited version. Keep a full paragraph or more. The box starts with the marked continuation.

In [ ]:
#@title Text box: run this cell
import ipywidgets as widgets
from IPython.display import display
text_box = widgets.Textarea(value=marked_text,
    layout=widgets.Layout(width="100%", height="180px"))
check_button = widgets.Button(description="Check text")
result = widgets.Output()
def on_check_clicked(button):
    result.clear_output()
    with result:
        print(check_watermark(text_box.value))
check_button.on_click(on_check_clicked)
display(text_box, check_button, result)

**Detected** means this text contains evidence of our watermark rule. **Not detected** does not establish that it came from the unmodified model. It could be human writing, another model, or edited marked text. False matches and missed marks are possible.

If the text box does not appear, the cell below performs the same check. Paste your continuation between the three quotation marks and run it:

In [ ]:
text_to_check = """Paste your generated continuation here."""
print(check_watermark(text_to_check))

## One more experiment
Change only the watermark `bias` from `2.0` to `0.0`. Rerun the watermark configuration, both marked generations, the comparison, and the detector cell.

With the same seed, beginning and generation settings, the zero-bias output should match the ordinary output. You removed the preference.

If you changed the seed earlier, rerun both ordinary generations before this comparison too. If you change `NEW_TOKENS`, rerun the settings cell before generating.

That's the connection to Part 1: the model gives us probabilities; our wrapper changes the preference before the next token is chosen.

[Generation and watermarking](https://huggingface.co/docs/transformers/v4.51.3/en/generation_features#watermarking) · [Repeatability](https://docs.pytorch.org/docs/stable/notes/randomness.html) · [Model](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct)